# `app_pages/chat.py`

Notebook version of `app_pages/chat.py`. The `.py` file is the source of truth (the app and the other scripts import it); after editing it, regenerate this notebook with `python tools/py_to_ipynb.py app_pages/chat.py --sync`.

In [ ]:
# Notebook setup: make the project root importable and define __file__,
# so the code behaves as if run as a script (run Jupyter from this file's folder).
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
__file__ = str(ROOT / 'app_pages/chat.py')

This file is a **Streamlit page**. Run the app with `streamlit run app.py` from the project root; executing the cells here only runs the page logic without a UI.

In [ ]:
import streamlit as st

from mofchat import theme

from mofchat.ui import get_agent, materials, render_response

INTRO = ("Ask for MOFs in plain English. Every number shown comes from the 12,020-MOF CoRE-MOF database; "
         "the plan, SQL and validation are shown under each answer.")

STEPS = [
    ("Plan", "An LLM (Gemini, Groq, OpenRouter or local Ollama) or the rule parser turns your question "
             "into a JSON plan, never code."),
    ("Validate", "Columns, units, physical ranges and every number in the plan are checked; errors go back "
                 "to the LLM."),
    ("Compute", "Read-only SQL, NSGA-II ranking, XGBoost + SHAP, FT-Transformer similarity or RAG over the "
                "ChatMOF paper."),
    ("Verify", "MOF IDs, numbers and comparison claims in the answer must match the results before it is shown."),
]

if st.session_state.messages:
    theme.page_header("MOFChat", "Conversational MOF recommender", INTRO)
else:
    df = materials()
    ready, total = st.session_state.get("models_ready", (0, 0))
    theme.hero("MOF", "Chat", "CoRE-MOF 2019 · explainable recommendation", INTRO, [
        (f"{len(df):,}", "MOFs"),
        (str(len({m for s in df.Metal_Type for m in s.split(",") if m})), "metals"),
        (f"{ready}/{total}", "language models ready"),
    ])

SUGGESTIONS = {
    ":material/filter_alt: Surface area > 4000 and void fraction > 0.8":
        "Find me 3 MOFs with a surface area greater than 4000 and a void fraction greater than 0.8",
    ":material/co2: Best MOFs for CO2 capture": "Recommend MOFs for CO2 capture",
    ":material/balance: Zinc MOFs: high surface area, low density":
        "Which zinc MOFs have high surface area and low density?",
    ":material/compare_arrows: MOFs similar to HKUST-1": "Find MOFs similar to HKUST-1",
    ":material/science: Separate CO2 from N2": "Which MOFs could separate CO2 from N2 by size?",
    ":material/help: What is pore limiting diameter?": "What is the pore limiting diameter and why does it matter?",
}

In [ ]:
agent = get_agent(st.session_state.get("use_llm", True), st.session_state.get("planner_mode", "hybrid"),
                  st.session_state.get("model_id"))

show_details = st.session_state.get("show_details", True)

prompt = st.chat_input("e.g. Cu MOFs with open metal sites and PLD between 4 and 6 Å", submit_mode="disable")

if not st.session_state.messages:
    theme.section_label("Try asking")
    picked = st.pills("Try asking", list(SUGGESTIONS), label_visibility="collapsed", key="suggestion")
    if picked:
        prompt = SUGGESTIONS[picked]
    theme.steps(STEPS)

if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.rerun()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        if msg["role"] == "user":
            st.markdown(msg["content"])
        else:
            render_response(msg["content"], show_details)

if st.session_state.messages and st.session_state.messages[-1]["role"] == "user":
    with st.chat_message("assistant"):
        with st.spinner("Planning, querying the database and checking the answer…"):
            response = agent.answer(st.session_state.messages[-1]["content"])
        render_response(response, show_details)
    st.session_state.messages.append({"role": "assistant", "content": response})